# Embodied Fly: Connectome-Driven Drosophila Locomotion

Interactive walkthrough of the connectome-to-behavior pipeline:
1. Build a spiking CPG network
2. Connect it to a virtual fly body
3. Analyze the resulting gait

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from cpg.spiking_cpg import SpikingCPG, CPGConfig
from cpg.connectome_cpg import ConnectomeCPG, ConnectomeConfig
from embodiment.neural_controller import NeuralController, SineWaveController
from analysis.neural_analysis import NeuralAnalyzer
from analysis.gait_analysis import GaitAnalyzer

## 1. Spiking CPG Network

Build a small network of 120 LIF neurons (20 per leg module) that generates
rhythmic motor patterns via half-center oscillators.

In [ ]:
# Initialize connectome-inspired CPG
cpg = ConnectomeCPG()
print(f"Network: {cpg.N} neurons")
print(f"Neuron types: {cpg.get_neuron_type_summary()}")
print(f"Weight stats: {cpg.get_weight_statistics()}")

In [ ]:
# Run for 2 seconds
duration_ms = 2000.0
spikes = cpg.run(duration_ms)
spike_np = spikes.cpu().numpy()
print(f"Spike matrix: {spike_np.shape} (timesteps x neurons)")
print(f"Total spikes: {spike_np.sum():.0f}")
print(f"Mean rate: {spike_np.sum() / (duration_ms * 1e-3 * cpg.N):.1f} Hz")

In [ ]:
# Spike raster plot
analyzer = NeuralAnalyzer(cpg.get_spike_history(), dt_ms=0.1, neurons_per_module=20)
fig = analyzer.plot_raster(time_range=(500, 1500))
plt.show()

In [ ]:
# Firing rates per module
fig = analyzer.plot_firing_rates(sigma_ms=20.0)
plt.show()

In [ ]:
# Phase relationships between contralateral legs
fig = analyzer.plot_phase_relationships()
plt.show()

## 2. Neural Controller: Spikes to Joint Angles

Decode CPG spike trains into continuous joint angle targets using
push-pull decoding.

In [ ]:
# Initialize controller and decode spikes
controller = NeuralController(cpg)
joint_targets = controller.decode_spike_batch(cpg.get_spike_history())

print(f"Joint targets shape: {joint_targets.shape}")

# Plot joint angle targets for the first leg (L1)
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
time_ms = np.arange(joint_targets.shape[0]) * 0.1

joint_names = ['Coxa', 'Coxa_roll', 'Coxa_yaw', 'Femur', 'Femur_roll', 'Tibia', 'Tarsus']
for j in range(3):
    axes[0].plot(time_ms, joint_targets[:, j], label=joint_names[j], alpha=0.8)
axes[0].set_ylabel('Angle (rad)')
axes[0].set_title('L1 Joint Targets (Coxa group)')
axes[0].legend()

for j in range(3, 5):
    axes[1].plot(time_ms, joint_targets[:, j], label=joint_names[j], alpha=0.8)
axes[1].set_ylabel('Angle (rad)')
axes[1].set_title('L1 Joint Targets (Femur group)')
axes[1].legend()

for j in range(5, 7):
    axes[2].plot(time_ms, joint_targets[:, j], label=joint_names[j], alpha=0.8)
axes[2].set_ylabel('Angle (rad)')
axes[2].set_xlabel('Time (ms)')
axes[2].set_title('L1 Joint Targets (Tibia/Tarsus)')
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Comparison: CPG vs Sine Wave

Compare the CPG-driven joint targets with a simple sine wave controller.

In [ ]:
# Generate sine wave targets for the same duration
sine_ctrl = SineWaveController(frequency=10.0, dt_ms=0.1)
n_steps = int(duration_ms / 0.1)
sine_targets = np.array([sine_ctrl.step() for _ in range(n_steps)])

# Compare L1 Coxa joint
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(time_ms, joint_targets[:, 0], 'b-', linewidth=0.8, label='CPG')
ax1.set_ylabel('L1 Coxa (rad)')
ax1.set_title('Spiking CPG Output')
ax1.legend()

ax2.plot(time_ms, sine_targets[:, 0], 'r-', linewidth=0.8, label='Sine')
ax2.set_ylabel('L1 Coxa (rad)')
ax2.set_xlabel('Time (ms)')
ax2.set_title('Sine Wave Output')
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Weight Matrix Visualization

Visualize the connectivity structure of the CPG network.

In [ ]:
W = cpg.W.numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Full weight matrix
im = ax1.imshow(W, cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)
ax1.set_title('CPG Weight Matrix')
ax1.set_xlabel('Source neuron')
ax1.set_ylabel('Target neuron')
fig.colorbar(im, ax=ax1, shrink=0.8)

# Add module boundaries
for i in range(1, 6):
    ax1.axhline(i * 20 - 0.5, color='black', linewidth=0.5)
    ax1.axvline(i * 20 - 0.5, color='black', linewidth=0.5)

# Weight distribution
nonzero = W[W != 0]
ax2.hist(nonzero[nonzero > 0], bins=30, alpha=0.7, color='red', label='Excitatory')
ax2.hist(nonzero[nonzero < 0], bins=30, alpha=0.7, color='blue', label='Inhibitory')
ax2.set_xlabel('Weight')
ax2.set_ylabel('Count')
ax2.set_title('Weight Distribution')
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Cross-Correlograms

Verify anti-phase coordination between contralateral leg pairs.

In [ ]:
fig = analyzer.plot_cross_correlograms(max_lag_ms=200.0)
plt.show()

## Next Steps

To run the full embodied simulation with FlyGym:
```bash
pip install flygym
python scripts/run_simulation.py --duration 2.0
python scripts/compare_gaits.py
python scripts/parameter_sweep.py
```